# Feature Prioritization Agent

An orchestrator agent prioritises three product features by consulting three **stakeholder agents** (User Advocate, Tech Lead, Business Analyst). The design follows one rule: **the LLMs reason, code decides.**

| Upgrade | What it does | Why it matters |
|---|---|---|
| 1. Real stakeholder agents | Each persona reads the feature brief and the customer signals, then returns a **Pydantic-validated** score, rationale and cited signal IDs | Scores come from reasoning over evidence, not a lookup table. Citing a signal that doesn't exist fails validation and triggers a retry |
| 2. Scoring in code | A `rank_features` tool computes weighted scores and tests whether the ranking holds under four weightings | The LLM never does the arithmetic, so the ranking is reproducible and auditable |
| 3. Disagreement → human review | If stakeholders' scores for a feature are far apart, the feature is flagged for a person instead of being averaged away | Mirrors human-in-the-loop controls for decisions that matter |
| 4. Offline tests | Scripted models test the scoring, schema retries, unknown tool calls and the step limit without any API calls | The control logic is proven before a single token is spent |

The agent loop is hand-written (no framework), to show what LangGraph or CrewAI do underneath: model → tool calls → results → repeat, with a step limit, error handling and parallel tool execution.

> The product, signals and estimates below are a **fictional scenario** for demonstration.

**Setup:** add `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` to Colab **Secrets** (🔑) with notebook access on, then set `PROVIDER`. The offline tests need no key.

## Install packages

In [38]:
!pip install -q -U langchain-core langchain-openai langchain-anthropic pydantic

## The scenario: features and customer signals

Every score a stakeholder gives must cite at least one of these signal IDs.

In [39]:
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import Literal

from pydantic import BaseModel, Field, ValidationError, field_validator
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

FEATURES = {
  "Dark Mode": "A dark colour theme across the web and mobile apps.",
  "AI-based Smart Search": "Semantic search across tasks, comments and files in all of a team's workspaces.",
  "Offline Support": "Create and update tasks on mobile without a connection, syncing when back online.",
}
FeatureName = Literal["Dark Mode", "AI-based Smart Search", "Offline Support"]
Perspective = Literal["user", "tech", "business"]
FEATURE_NAMES = tuple(FEATURES)
PERSPECTIVES = ("user", "tech", "business")

SIGNALS = {
  "S1": "Support tickets (last quarter): 38 tickets ask for better search across projects, e.g. 'I can never find old tasks'.",
  "S2": "User interview (field sales rep): 'On site visits I lose signal and can't update my tasks until I'm back at the office.'",
  "S3": "NPS comments: 6 of 120 comments mention wanting dark mode.",
  "S4": "Engineering note: a keyword search index already exists; semantic search needs an embedding service and an evaluation set. Estimate: 6 weeks.",
  "S5": "Engineering note: offline mode needs local storage and sync conflict resolution. Estimate: 8 weeks; the mobile team is at capacity this quarter.",
  "S6": "Engineering note: the design system already supports theme tokens. Dark mode estimate: 1 week.",
  "S7": "Sales: 3 enterprise deals in the pipeline list 'search across workspaces' as a requirement.",
  "S8": "Churn survey: 12% of churned teams cite 'hard to find information'.",
  "S9": "Competitor note: the main competitor shipped AI search last month.",
}

for sid, text in SIGNALS.items():
  print(f"{sid}: {text}")

S1: Support tickets (last quarter): 38 tickets ask for better search across projects, e.g. 'I can never find old tasks'.
S2: User interview (field sales rep): 'On site visits I lose signal and can't update my tasks until I'm back at the office.'
S3: NPS comments: 6 of 120 comments mention wanting dark mode.
S4: Engineering note: a keyword search index already exists; semantic search needs an embedding service and an evaluation set. Estimate: 6 weeks.
S5: Engineering note: offline mode needs local storage and sync conflict resolution. Estimate: 8 weeks; the mobile team is at capacity this quarter.
S6: Engineering note: the design system already supports theme tokens. Dark mode estimate: 1 week.
S7: Sales: 3 enterprise deals in the pipeline list 'search across workspaces' as a requirement.
S8: Churn survey: 12% of churned teams cite 'hard to find information'.
S9: Competitor note: the main competitor shipped AI search last month.


## Stakeholder agents with validated, evidence-backed output

Each stakeholder is an LLM call with its own persona and scoring rubric. Its reply must be JSON matching the `Assessment` schema:

- `score` must be an integer from 1 to 10.
- `evidence` must cite **only signal IDs that exist**: a made-up `S42` fails validation.

If validation fails, `call_structured` sends the exact error back to the model and asks for corrected JSON, up to `max_retries` times.

In [40]:
class Assessment(BaseModel):
  feature: FeatureName
  perspective: Perspective
  score: int = Field(ge=1, le=10)
  rationale: str = Field(min_length=10, max_length=500)
  evidence: list[str] = Field(min_length=1)

  @field_validator("evidence")
  @classmethod
  def evidence_must_exist(cls, ids):
      unknown = [i for i in ids if i not in SIGNALS]
      if unknown:
          raise ValueError(f"cites signal IDs that don't exist: {unknown}. Valid IDs: {list(SIGNALS)}")
      return ids


PERSONAS = {
  "user": ("User Advocate",
            "user desirability: how many users want this and how much it helps them. "
            "10 = strong, widespread demand; 1 = almost nobody asked for it."),
  "tech": ("Tech Lead",
            "technical feasibility THIS QUARTER: higher means easier. "
            "10 = trivial and fits easily; 1 = very hard or no team capacity. Weigh effort estimates and capacity."),
  "business": ("Business Analyst",
                "business value: revenue, retention and competitive position. "
                "10 = major impact; 1 = negligible."),
}


def persona_messages(perspective: str, feature: str) -> list:
  role, rubric = PERSONAS[perspective]
  signals = "\n".join(f"{sid}: {text}" for sid, text in SIGNALS.items())
  system = (
    f"You are the {role} on a product team. You score features ONLY on {rubric}\n"
    "Base your score on the signals provided and cite the signal IDs you relied on. "
    "Do not invent evidence."
  )
  human = (
    f"Feature: {feature}\n"
    f"Perspective: {perspective}\n"
    f"Description: {FEATURES[feature]}\n\n"
    f"Signals:\n{signals}\n\n"
    "Reply with ONLY a JSON object, no other text:\n"
    '{"feature": "<feature>", "perspective": "<perspective>", "score": <1-10>, '
    '"rationale": "<one or two sentences>", "evidence": ["<signal IDs>"]}'
  )
  return [SystemMessage(content=system), HumanMessage(content=human)]


def text_of(message) -> str:
  """OpenAI returns a string; Anthropic can return a list of content blocks."""
  if isinstance(message.content, str):
    return message.content
  return "".join(b.get("text", "") for b in message.content if isinstance(b, dict))


def extract_json(text: str) -> str:
  match = re.search(r"\{.*\}", text, re.DOTALL)
  if not match:
    raise ValueError("no JSON object found in the reply")
  return match.group(0)


def call_structured(model, messages, schema, max_retries: int = 2):
  """Call the model, validate against `schema`, and on failure retry with the validation error."""
  messages = list(messages)
  for attempt in range(max_retries + 1):
    raw = text_of(model.invoke(messages))
    try:
      return schema.model_validate_json(extract_json(raw)), attempt
    except (ValidationError, ValueError) as err:
      if attempt == max_retries:
          raise
      messages += [
          AIMessage(content=raw),
          HumanMessage(content=f"Your reply was invalid:\n{err}\nReply with ONLY the corrected JSON object."),
      ]

##Tools for the orchestrator, with scoring in code

The orchestrator agent gets two tools:

- `get_stakeholder_assessment(feature, perspective)` runs one stakeholder agent and stores its validated assessment.
- `rank_features(...)` computes the ranking **in Python**: weighted scores, a sensitivity check across four weightings, and disagreement flags. It refuses to rank until all nine assessments exist.

A feature is flagged for **human review** when the gap between its highest and lowest stakeholder score is at least `DISAGREEMENT_THRESHOLD`.

In [41]:
CTX = {"stakeholder_llm": None}   # set in Step 5 (or swapped for a scripted model in the tests)
ASSESSMENTS: dict = {}            # (feature, perspective) -> Assessment
RETRY_LOG: list = []              # (feature, perspective, retries needed)
LAST_RANKING: dict = {}

DISAGREEMENT_THRESHOLD = 5
WEIGHT_SCENARIOS = {
  "balanced":          {"user": 1, "tech": 1, "business": 1},
  "user-first":        {"user": 2, "tech": 1, "business": 1},
  "feasibility-first": {"user": 1, "tech": 2, "business": 1},
  "business-first":    {"user": 1, "tech": 1, "business": 2},
}


def weighted_score(scores: dict, weights: dict) -> float:
  return round(sum(scores[p] * weights[p] for p in PERSPECTIVES) / sum(weights.values()), 2)


def order_features(scores: dict, weights: dict) -> list:
  """Highest weighted score first; ties broken alphabetically so results are reproducible."""
  return sorted(FEATURE_NAMES, key=lambda f: (-weighted_score(scores[f], weights), f))


def compute_ranking(assessments: dict, weights: dict | None = None) -> dict:
  missing = [f"{f} / {p}" for f in FEATURE_NAMES for p in PERSPECTIVES if (f, p) not in assessments]
  if missing:
    return {"status": "incomplete", "missing": missing,
            "instruction": "Get these assessments before ranking."}

  weights = weights or WEIGHT_SCENARIOS["balanced"]
  scores = {f: {p: assessments[(f, p)].score for p in PERSPECTIVES} for f in FEATURE_NAMES}

  ranking = [
    {"rank": i + 1, "feature": f, "weighted_score": weighted_score(scores[f], weights), **scores[f]}
    for i, f in enumerate(order_features(scores, weights))
  ]
  sensitivity = {name: order_features(scores, w) for name, w in WEIGHT_SCENARIOS.items()}
  top_choice_stable = len({order[0] for order in sensitivity.values()}) == 1

  needs_human_review = []
  for f in FEATURE_NAMES:
    s = scores[f]
    spread = max(s.values()) - min(s.values())
    if spread >= DISAGREEMENT_THRESHOLD:
      high, low = max(s, key=s.get), min(s, key=s.get)
      needs_human_review.append({
          "feature": f, "spread": spread,
          "reason": f"{high} scored {s[high]} but {low} scored {s[low]}",
          "action": "human review before committing",
      })

  return {"status": "complete", "weights": weights, "ranking": ranking,
          "sensitivity": sensitivity, "top_choice_stable": top_choice_stable,
          "needs_human_review": needs_human_review}


@tool
def get_stakeholder_assessment(feature: FeatureName, perspective: Perspective) -> str:
  """Ask one stakeholder agent to score one feature from its perspective:
  'user' = desirability, 'tech' = feasibility this quarter (higher is easier), 'business' = business value.
  Returns a validated JSON assessment with a 1-10 score, a rationale and the signal IDs it cites."""

  if feature not in FEATURES or perspective not in PERSONAS:
    raise ValueError(f"feature must be one of {list(FEATURES)}; perspective one of {list(PERSONAS)}")
  assessment, retries = call_structured(CTX["stakeholder_llm"], persona_messages(perspective, feature), Assessment)
  # Pin the identity to what was asked, so a model can't file its answer under another feature.
  assessment = assessment.model_copy(update={"feature": feature, "perspective": perspective})
  ASSESSMENTS[(feature, perspective)] = assessment
  RETRY_LOG.append((feature, perspective, retries))
  return assessment.model_dump_json()


@tool
def rank_features(user_weight: float = 1.0, tech_weight: float = 1.0, business_weight: float = 1.0) -> str:
  """Rank all features from the stored stakeholder assessments. Scores are computed in code.
  Returns the ranking, a sensitivity check across weightings, and any features needing human review.
  Use the default weights unless the user asked for a different emphasis."""
  weights = {"user": user_weight, "tech": tech_weight, "business": business_weight}
  if any(w <= 0 for w in weights.values()):
    raise ValueError("all weights must be positive")
  result = compute_ranking(ASSESSMENTS, weights)
  if result["status"] == "complete":
    LAST_RANKING.clear()
    LAST_RANKING.update(result)
  return json.dumps(result)


TOOLS = [get_stakeholder_assessment, rank_features]
TOOL_BY_NAME = {t.name: t for t in TOOLS}
print("Tools:", list(TOOL_BY_NAME))

Tools: ['get_stakeholder_assessment', 'rank_features']


## Connect to LLM
Set `PROVIDER` to `"openai"` or `"anthropic"`. The orchestrator gets the tools; the stakeholder agents use the plain model.

In [42]:
import os

def _load(key):
  """Colab Secrets first, then an environment variable."""
  try:
    from google.colab import userdata
    value = userdata.get(key)
    if value:
      return value
  except Exception:
    pass
  return os.environ.get(key)

PROVIDER = "openai"   # or "anthropic"

if PROVIDER == "openai":
  from langchain_openai import ChatOpenAI
  llm = ChatOpenAI(model="gpt-5-mini", api_key=_load("OPENAI_API_KEY"))
elif PROVIDER == "anthropic":
  from langchain_anthropic import ChatAnthropic
  llm = ChatAnthropic(model="claude-haiku-5-5", api_key=_load("ANTHROPIC_API_KEY"), max_tokens=2048)
else:
  raise ValueError('PROVIDER must be "openai" or "anthropic"')

CTX["stakeholder_llm"] = llm
orchestrator = llm.bind_tools(TOOLS)
print(f"Using {PROVIDER}")

Using openai


## The agent loop

Compared with a basic loop, this one:

- returns an **error message to the model** for an unknown tool or bad arguments, instead of crashing;
- runs independent tool calls **in parallel** (the nine stakeholder assessments don't depend on each other);
- records a **trace** of every tool call (step, tool, arguments, success, time);
- stops at `max_steps` and says so.

In [43]:
ORCHESTRATOR_PROMPT = f"""You coordinate feature prioritisation for a product team.
Features: {", ".join(FEATURE_NAMES)}.
Stakeholder perspectives: user (desirability), tech (feasibility), business (value).

Process:
1. Call get_stakeholder_assessment for EVERY feature and perspective (9 calls). You may request several at once.
2. When all assessments exist, call rank_features (default weights unless the user asks otherwise).
3. Write a recommendation of under 200 words that:
   - follows the ranking returned by rank_features exactly. Never compute, change or re-order scores yourself;
   - explains the top choice using the stakeholders' rationales;
   - names EVERY feature listed in needs_human_review and states that it needs human review before commitment;
   - says whether the top choice holds across the different weightings.
"""


@dataclass
class AgentResult:
  text: str
  stopped: str
  trace: list


def execute_tool_call(call: dict):
  start = time.perf_counter()
  name, args = call["name"], call.get("args") or {}
  tool_fn = TOOL_BY_NAME.get(name)
  if tool_fn is None:
    output, ok = f"ERROR: unknown tool '{name}'. Available tools: {sorted(TOOL_BY_NAME)}.", False
  else:
    try:
      output, ok = str(tool_fn.invoke(args)), True
    except Exception as err:   # surface the problem to the model instead of crashing
      output, ok = f"ERROR: {type(err).__name__}: {err}", False
  entry = {"tool": name, "args": args, "ok": ok, "ms": round((time.perf_counter() - start) * 1000)}
  return ToolMessage(content=output, tool_call_id=call["id"]), entry


def run_agent(model, task: str, system: str = ORCHESTRATOR_PROMPT, max_steps: int = 8, parallel: bool = True, verbose: bool = True) -> AgentResult:
  messages = [SystemMessage(content=system), HumanMessage(content=task)]
  trace = []
  for step in range(1, max_steps + 1):
    response = model.invoke(messages)
    messages.append(response)
    calls = response.tool_calls or []
    if not calls:
      return AgentResult(text_of(response), "final_answer", trace)

    if parallel and len(calls) > 1:
      with ThreadPoolExecutor(max_workers=min(4, len(calls))) as pool:
        results = list(pool.map(execute_tool_call, calls))   # map keeps the original order
    else:
      results = [execute_tool_call(c) for c in calls]

    for message, entry in results:
      messages.append(message)
      trace.append({"step": step, **entry})
      if verbose:
        status = "ok " if entry["ok"] else "ERR"
        print(f"[step {step}] {status} {entry['tool']} {entry['args']} ({entry['ms']} ms)")
  return AgentResult("Stopped: reached max_steps without a final answer.", "max_steps", trace)


def check_narrative(text: str, ranking: dict) -> list:
  """Code-level consistency check: does the written recommendation match the computed ranking?"""
  if ranking.get("status") != "complete":
    return ["no complete ranking was computed"]
  problems = []
  top = ranking["ranking"][0]["feature"]
  if top not in text:
    problems.append(f"the recommendation never names the top-ranked feature ({top})")
  for flag in ranking["needs_human_review"]:
    if flag["feature"] not in text:
      problems.append(f"flagged feature not mentioned: {flag['feature']}")
  if ranking["needs_human_review"] and "review" not in text.lower():
    problems.append("flagged features exist but the recommendation never mentions human review")
  return problems


def render_ranking(ranking: dict) -> str:
  lines = ["| Rank | Feature | User | Tech (feasibility) | Business | Weighted score |",
            "|---|---|---|---|---|---|"]
  for r in ranking["ranking"]:
    lines.append(f"| {r['rank']} | {r['feature']} | {r['user']} | {r['tech']} | {r['business']} | {r['weighted_score']} |")
  lines.append("\nSensitivity (top to bottom under each weighting):")
  for name, order in ranking["sensitivity"].items():
    lines.append(f"  {name:<18} {' > '.join(order)}")
  lines.append(f"Top choice stable across weightings: {ranking['top_choice_stable']}")
  for flag in ranking["needs_human_review"]:
    lines.append(f"HUMAN REVIEW: {flag['feature']} ({flag['reason']}; spread {flag['spread']})")
  return "\n".join(lines)

## Offline tests: no API key, no cost

Scripted models stand in for the LLM, so the control logic is tested deterministically: the scoring maths, schema retries, an unknown tool call, the ranking guard and the step limit.

In [44]:
class ScriptedModel:
  """Returns pre-written replies in order: an offline stand-in for an LLM."""
  def __init__(self, replies):
    self.replies, self.calls, self.seen = list(replies), 0, []

  def invoke(self, messages):
    self.seen.append(list(messages))
    reply = self.replies[min(self.calls, len(self.replies) - 1)]
    self.calls += 1
    return reply if isinstance(reply, AIMessage) else AIMessage(content=reply)


class FakeStakeholder:
  """Answers from a fixed score table, reading the feature and perspective from the prompt.
  Safe to call from parallel threads because it doesn't depend on call order."""
  def __init__(self, table):
    self.table = table

  def invoke(self, messages):
    prompt = messages[-1].content
    feature = re.search(r"Feature: (.+)", prompt).group(1).strip()
    perspective = re.search(r"Perspective: (\w+)", prompt).group(1)
    score = self.table[(feature, perspective)]
    return AIMessage(content=json.dumps({
      "feature": feature, "perspective": perspective, "score": score,
      "rationale": f"Scripted {perspective} view of {feature}.", "evidence": ["S1"]}))


def reset_state():
  ASSESSMENTS.clear(); RETRY_LOG.clear(); LAST_RANKING.clear()


def call(name, args, i):
  return {"name": name, "args": args, "id": f"call_{i}", "type": "tool_call"}


TABLE = {
    ("Dark Mode", "user"): 5, ("Dark Mode", "tech"): 8, ("Dark Mode", "business"): 4,
    ("AI-based Smart Search", "user"): 9, ("AI-based Smart Search", "tech"): 5, ("AI-based Smart Search", "business"): 9,
    ("Offline Support", "user"): 8, ("Offline Support", "tech"): 2, ("Offline Support", "business"): 5,
}
_saved_llm = CTX["stakeholder_llm"]

# 1. Scoring maths, sensitivity and disagreement flags, checked against hand-computed values
reset_state()
for (f, p), s in TABLE.items():
    ASSESSMENTS[(f, p)] = Assessment(feature=f, perspective=p, score=s, rationale="hand-written test case", evidence=["S1"])
r = compute_ranking(ASSESSMENTS)
assert [x["feature"] for x in r["ranking"]] == ["AI-based Smart Search", "Dark Mode", "Offline Support"]
assert [x["weighted_score"] for x in r["ranking"]] == [7.67, 5.67, 5.0]
assert r["sensitivity"]["user-first"] == ["AI-based Smart Search", "Offline Support", "Dark Mode"]  # Offline overtakes Dark Mode
assert r["top_choice_stable"] is True
assert [f["feature"] for f in r["needs_human_review"]] == ["Offline Support"]   # user 8 vs tech 2
print("✓ 1. scoring, sensitivity and disagreement flag match hand-computed values")

# 2. Schema validation with retries: out-of-range score, then an invented signal ID, then valid
bad_score = '{"feature": "Dark Mode", "perspective": "user", "score": 15, "rationale": "Users love it a lot.", "evidence": ["S3"]}'
bad_evidence = '{"feature": "Dark Mode", "perspective": "user", "score": 5, "rationale": "Users love it a lot.", "evidence": ["S42"]}'
valid = '{"feature": "Dark Mode", "perspective": "user", "score": 5, "rationale": "Only 6 of 120 NPS comments ask for it.", "evidence": ["S3"]}'
model = ScriptedModel([bad_score, bad_evidence, valid])
result, retries = call_structured(model, persona_messages("user", "Dark Mode"), Assessment, max_retries=2)
assert result.score == 5 and retries == 2
assert "less than or equal to 10" in model.seen[1][-1].content      # the model was told what was wrong
assert "S42" in model.seen[2][-1].content
try:
    call_structured(ScriptedModel([bad_score]), persona_messages("user", "Dark Mode"), Assessment, max_retries=1)
    raise AssertionError("should have failed")
except ValidationError:
    pass
print("✓ 2. invalid output is retried with the error message; persistent failure raises")

# 3. Full agent run offline, including a hallucinated tool name
reset_state()
CTX["stakeholder_llm"] = FakeStakeholder(TABLE)
assessment_calls = [call("get_stakeholder_assessment", {"feature": f, "perspective": p}, i)
                    for i, (f, p) in enumerate(TABLE)]
orchestrator_script = ScriptedModel([
  AIMessage(content="", tool_calls=assessment_calls + [call("MarketResearch", {"query": "dark mode"}, 99)]),
  AIMessage(content="", tool_calls=[call("rank_features", {}, 100)]),
  AIMessage(content="Prioritise AI-based Smart Search: the top choice under every weighting. "
                    "Offline Support needs human review: user demand is high but feasibility is low this quarter."),
])
run = run_agent(orchestrator_script, "Prioritise the features.", verbose=False)
assert run.stopped == "final_answer"
assert sum(e["ok"] for e in run.trace if e["tool"] == "get_stakeholder_assessment") == 9
assert any(e["tool"] == "MarketResearch" and not e["ok"] for e in run.trace)   # handled, not crashed
unknown_reply = [m for m in orchestrator_script.seen[1] if isinstance(m, ToolMessage) and m.tool_call_id == "call_99"][0]
assert "unknown tool" in unknown_reply.content
assert LAST_RANKING["ranking"][0]["feature"] == "AI-based Smart Search"
assert check_narrative(run.text, LAST_RANKING) == []
assert check_narrative("We should build Dark Mode first.", LAST_RANKING) != []   # an inconsistent write-up is caught
print("✓ 3. full run: 9 assessments in parallel, unknown tool reported back to the model, narrative matches ranking")

# 4. rank_features refuses to rank before all assessments exist
reset_state()
partial = json.loads(rank_features.invoke({}))
assert partial["status"] == "incomplete" and len(partial["missing"]) == 9
print("✓ 4. ranking is blocked until all 9 assessments exist")

# 5. The step limit stops a model that never finishes
looping = ScriptedModel([AIMessage(content="", tool_calls=[call("rank_features", {}, 1)])])
stuck = run_agent(looping, "Prioritise the features.", max_steps=3, verbose=False)
assert stuck.stopped == "max_steps" and looping.calls == 3
print("✓ 5. max_steps stops an endless tool-calling loop")

reset_state()
CTX["stakeholder_llm"] = _saved_llm
print("\nAll offline tests passed.")

✓ 1. scoring, sensitivity and disagreement flag match hand-computed values
✓ 2. invalid output is retried with the error message; persistent failure raises
✓ 3. full run: 9 assessments in parallel, unknown tool reported back to the model, narrative matches ranking
✓ 4. ranking is blocked until all 9 assessments exist
✓ 5. max_steps stops an endless tool-calling loop

All offline tests passed.


## Run the agent live

In [36]:
reset_state()
TASK = "Prioritise Dark Mode, AI-based Smart Search and Offline Support for next quarter."
start = time.perf_counter()
run = run_agent(orchestrator, TASK)
elapsed = time.perf_counter() - start
print(f"\nStopped because: {run.stopped}  |  {len(run.trace)} tool calls  |  {elapsed:.1f} s")

[step 1] ok  get_stakeholder_assessment {'feature': 'Dark Mode', 'perspective': 'user'} (6327 ms)
[step 1] ok  get_stakeholder_assessment {'feature': 'Dark Mode', 'perspective': 'tech'} (6540 ms)
[step 1] ok  get_stakeholder_assessment {'feature': 'Dark Mode', 'perspective': 'business'} (7441 ms)
[step 1] ok  get_stakeholder_assessment {'feature': 'AI-based Smart Search', 'perspective': 'user'} (7759 ms)
[step 1] ok  get_stakeholder_assessment {'feature': 'AI-based Smart Search', 'perspective': 'tech'} (9337 ms)
[step 1] ok  get_stakeholder_assessment {'feature': 'AI-based Smart Search', 'perspective': 'business'} (7533 ms)
[step 1] ok  get_stakeholder_assessment {'feature': 'Offline Support', 'perspective': 'user'} (6125 ms)
[step 1] ok  get_stakeholder_assessment {'feature': 'Offline Support', 'perspective': 'tech'} (4956 ms)
[step 1] ok  get_stakeholder_assessment {'feature': 'Offline Support', 'perspective': 'business'} (12979 ms)
[step 2] ok  rank_features {'user_weight': 1.0, 'te

## Results

The table and flags below are computed **by code**. The recommendation underneath is the agent's write-up, checked against them.

In [37]:
if LAST_RANKING:
  print(render_ranking(LAST_RANKING))
  print("\nStakeholder rationales:")
  for (f, p), a in sorted(ASSESSMENTS.items()):
    print(f"  {f} / {p}: {a.score} because {a.rationale} (evidence: {', '.join(a.evidence)})")

print("\n" + "=" * 70 + "\nAGENT RECOMMENDATION\n" + "=" * 70)
print(run.text)

problems = check_narrative(run.text, LAST_RANKING)
print("\nConsistency check:", "passed" if not problems else "FAILED")
for p in problems:
  print("  -", p)

errors = [e for e in run.trace if not e["ok"]]
retried = [r for r in RETRY_LOG if r[2] > 0]
print(f"\nTrace: {len(run.trace)} tool calls, {len(errors)} errors, {len(retried)} assessments needed a retry")

| Rank | Feature | User | Tech (feasibility) | Business | Weighted score |
|---|---|---|---|---|---|
| 1 | AI-based Smart Search | 9 | 6 | 8 | 7.67 |
| 2 | Offline Support | 5 | 1 | 6 | 4.0 |
| 3 | Dark Mode | 3 | 6 | 2 | 3.67 |

Sensitivity (top to bottom under each weighting):
  balanced           AI-based Smart Search > Offline Support > Dark Mode
  user-first         AI-based Smart Search > Offline Support > Dark Mode
  feasibility-first  AI-based Smart Search > Dark Mode > Offline Support
  business-first     AI-based Smart Search > Offline Support > Dark Mode
Top choice stable across weightings: True
HUMAN REVIEW: Offline Support (business scored 6 but tech scored 1; spread 5)

Stakeholder rationales:
  AI-based Smart Search / business: 8 because This feature targets clear product gaps and retention risks (38 support tickets requesting better search; 12% of churn cites hard-to-find information) and is cited as a requirement in enterprise pipeline deals, while a competitor has alr

## Design notes

- **The model reasons; code decides.** Stakeholder agents judge evidence, but the ranking, sensitivity and flags are computed in Python and are reproducible. The final write-up is then checked against the computed ranking.
- **Grounding is enforced, not requested.** Every score must cite real signal IDs, and an invented ID fails validation and is retried.
- **Disagreement is surfaced, not averaged away.** A feature that users want but engineering can't build this quarter is a decision for a person.
- **Failures are contained.** An unknown tool, bad arguments or an invalid reply becomes an error message the model can recover from. The step limit stops runaway loops.
